# 🔧 Data Preprocessing
## Credit Scoring - Give Me Some Credit

**Author**: QuangMinh  
**Course**: MLE501 - AI & Machine Learning  
**Date**: 2026-03-10

---

## 🎯 Objectives

Based on EDA findings, this notebook will:

1. ✅ **Handle Missing Values** (MonthlyIncome: 19.82%, NumberOfDependents: 2.62%)
2. ✅ **Handle Outliers** (DebtRatio: 20.87%, Late payments: 15.99%)
3. ✅ **Feature Engineering** (+7 features mới từ domain knowledge)
4. ✅ **Train-Validation Split** (Stratified 80/20)
5. ✅ **Feature Scaling** (StandardScaler for Logistic Regression & SVM)
6. ✅ **Save Processed Data & Objects**

**Ghi chu**: Imbalanced data (13.96:1) se duoc xu ly trong Phase 3 bang `class_weight='balanced'` / `scale_pos_weight` thay vi SMOTE, de tranh tao mau tong hop co the gay nhieu.

## 📦 Import Libraries

In [1]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Preprocessing
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.model_selection import train_test_split

# Model persistence
import joblib
import pickle

# Utilities
import warnings
from datetime import datetime
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

# Plot settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✅ Libraries imported successfully!")
print(f"Preprocessing started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ Libraries imported successfully!
Preprocessing started at: 2026-03-14 22:11:02


## 📂 Load Data

In [2]:
# Load training and test data
train_df = pd.read_csv('../data/raw/cs-training.csv')
test_df = pd.read_csv('../data/raw/cs-test.csv')

# Drop unnamed index column if exists
if 'Unnamed: 0' in train_df.columns:
    train_df = train_df.drop('Unnamed: 0', axis=1)
    test_df = test_df.drop('Unnamed: 0', axis=1)

print(f"Training data shape: {train_df.shape}")
print(f"Test data shape: {test_df.shape}")
print(f"\nColumns: {train_df.columns.tolist()}")

Training data shape: (150000, 11)
Test data shape: (101503, 11)

Columns: ['SeriousDlqin2yrs', 'RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents']


In [3]:
# Separate features and target
target_col = 'SeriousDlqin2yrs'

X_train = train_df.drop(target_col, axis=1)
y_train = train_df[target_col]

X_test = test_df.drop(target_col, axis=1) if target_col in test_df.columns else test_df.copy()

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")

# Store original copies for comparison
X_train_original = X_train.copy()
X_test_original = X_test.copy()

print("\n✅ Data loaded and split successfully!")

X_train shape: (150000, 10)
y_train shape: (150000,)
X_test shape: (101503, 10)

✅ Data loaded and split successfully!


## 1️⃣ Handle Missing Values

### EDA Findings:
- **MonthlyIncome**: 19.82% missing (29,731 values)
- **NumberOfDependents**: 2.62% missing (3,924 values)

### Strategy:
- **MonthlyIncome**: Median imputation (right-skewed distribution)
- **NumberOfDependents**: Median imputation (discrete values)

In [4]:
# Check missing values before
print("="*80)
print("MISSING VALUES - BEFORE IMPUTATION")
print("="*80)

missing_before_train = X_train.isnull().sum()
missing_before_test = X_test.isnull().sum()

missing_summary_before = pd.DataFrame({
    'Feature': X_train.columns,
    'Train_Missing': missing_before_train.values,
    'Train_Pct': (missing_before_train.values / len(X_train) * 100).round(2),
    'Test_Missing': missing_before_test.values,
    'Test_Pct': (missing_before_test.values / len(X_test) * 100).round(2)
})

missing_summary_before = missing_summary_before[
    (missing_summary_before['Train_Missing'] > 0) | 
    (missing_summary_before['Test_Missing'] > 0)
]

if len(missing_summary_before) > 0:
    display(missing_summary_before)
else:
    print("No missing values found!")

MISSING VALUES - BEFORE IMPUTATION


,Feature,Train_Missing,Train_Pct,Test_Missing,Test_Pct
4,MonthlyIncome,29731,19.8200,20103,19.8100
9,NumberOfDependents,3924,2.6200,2626,2.5900


In [5]:
# Impute MonthlyIncome with median
print("\n" + "="*80)
print("IMPUTING MONTHLY INCOME (Median Strategy)")
print("="*80)

income_imputer = SimpleImputer(strategy='median')

# Fit on training data
X_train['MonthlyIncome'] = income_imputer.fit_transform(X_train[['MonthlyIncome']])
X_test['MonthlyIncome'] = income_imputer.transform(X_test[['MonthlyIncome']])

imputed_value_income = income_imputer.statistics_[0]
print(f"✅ Imputed value (median): ${imputed_value_income:,.2f}")
print(f"   Training - Missing before: {missing_before_train['MonthlyIncome']}")
print(f"   Training - Missing after: {X_train['MonthlyIncome'].isnull().sum()}")
print(f"   Test - Missing before: {missing_before_test['MonthlyIncome']}")
print(f"   Test - Missing after: {X_test['MonthlyIncome'].isnull().sum()}")


IMPUTING MONTHLY INCOME (Median Strategy)
✅ Imputed value (median): $5,400.00
   Training - Missing before: 29731
   Training - Missing after: 0
   Test - Missing before: 20103
   Test - Missing after: 0


In [6]:
# Impute NumberOfDependents with median
print("\n" + "="*80)
print("IMPUTING NUMBER OF DEPENDENTS (Median Strategy)")
print("="*80)

dependents_imputer = SimpleImputer(strategy='median')

X_train['NumberOfDependents'] = dependents_imputer.fit_transform(X_train[['NumberOfDependents']])
X_test['NumberOfDependents'] = dependents_imputer.transform(X_test[['NumberOfDependents']])

imputed_value_dependents = dependents_imputer.statistics_[0]
print(f"✅ Imputed value (median): {imputed_value_dependents:.0f}")
print(f"   Training - Missing before: {missing_before_train['NumberOfDependents']}")
print(f"   Training - Missing after: {X_train['NumberOfDependents'].isnull().sum()}")
print(f"   Test - Missing before: {missing_before_test['NumberOfDependents']}")
print(f"   Test - Missing after: {X_test['NumberOfDependents'].isnull().sum()}")


IMPUTING NUMBER OF DEPENDENTS (Median Strategy)
✅ Imputed value (median): 0
   Training - Missing before: 3924
   Training - Missing after: 0
   Test - Missing before: 2626
   Test - Missing after: 0


In [7]:
# Verify no missing values remain
print("\n" + "="*80)
print("VERIFICATION - AFTER IMPUTATION")
print("="*80)

missing_after_train = X_train.isnull().sum().sum()
missing_after_test = X_test.isnull().sum().sum()

print(f"Training - Total missing values: {missing_after_train}")
print(f"Test - Total missing values: {missing_after_test}")

if missing_after_train == 0 and missing_after_test == 0:
    print("\n✅ SUCCESS: All missing values have been imputed!")
else:
    print("\n⚠️  WARNING: Some missing values remain!")
    print(X_train.isnull().sum()[X_train.isnull().sum() > 0])
    print(X_test.isnull().sum()[X_test.isnull().sum() > 0])


VERIFICATION - AFTER IMPUTATION
Training - Total missing values: 0
Test - Total missing values: 0

✅ SUCCESS: All missing values have been imputed!


## 2️⃣ Handle Outliers

### EDA Findings (Outliers by IQR method):
- **DebtRatio**: 20.87% outliers → Cap at 99th percentile
- **NumberOfTime30-59DaysPastDueNotWorse**: 15.99% → Cap at 15
- **NumberOfDependents**: 9.13% → Cap at 10
- **NumberOfTimes90DaysLate**: 5.56% → Cap at 15
- **NumberOfTime60-89DaysPastDueNotWorse**: 5.07% → Cap at 15

### Strategy: Capping (Winsorization)

In [8]:
# Define capping rules
capping_rules = {
    'age': {'min': 18, 'max': 100},
    'DebtRatio': {'min': None, 'max': 5},  # Fixed: was percentile 99 (=4979), now max 5
    'RevolvingUtilizationOfUnsecuredLines': {'min': None, 'max': 2.0},
    'NumberOfTime30-59DaysPastDueNotWorse': {'min': None, 'max': 15},
    'NumberOfTimes90DaysLate': {'min': None, 'max': 15},
    'NumberOfTime60-89DaysPastDueNotWorse': {'min': None, 'max': 15},
    'NumberOfDependents': {'min': None, 'max': 10}
}

print("="*80)
print("OUTLIER HANDLING - CAPPING STRATEGY")
print("="*80)
print("\nCapping Rules:")
for feature, rules in capping_rules.items():
    print(f"  • {feature}: {rules}")

OUTLIER HANDLING - CAPPING STRATEGY

Capping Rules:
  • age: {'min': 18, 'max': 100}
  • DebtRatio: {'min': None, 'max': 5}
  • RevolvingUtilizationOfUnsecuredLines: {'min': None, 'max': 2.0}
  • NumberOfTime30-59DaysPastDueNotWorse: {'min': None, 'max': 15}
  • NumberOfTimes90DaysLate: {'min': None, 'max': 15}
  • NumberOfTime60-89DaysPastDueNotWorse: {'min': None, 'max': 15}
  • NumberOfDependents: {'min': None, 'max': 10}


In [9]:
# Function to cap outliers
def cap_outliers(df, feature, min_val=None, max_val=None, percentile=None):
    """
    Cap outliers in a feature
    """
    original_values = df[feature].copy()
    
    # Calculate percentile if specified
    if percentile is not None:
        max_val = df[feature].quantile(percentile / 100)
    
    # Cap minimum
    if min_val is not None:
        below_min = (df[feature] < min_val).sum()
        df[feature] = df[feature].clip(lower=min_val)
    else:
        below_min = 0
    
    # Cap maximum
    if max_val is not None:
        above_max = (original_values > max_val).sum()
        df[feature] = df[feature].clip(upper=max_val)
    else:
        above_max = 0
    
    total_capped = below_min + above_max
    pct_capped = (total_capped / len(df)) * 100
    
    return below_min, above_max, total_capped, pct_capped, max_val if percentile else None

# Apply capping
outlier_summary = []

print("\n" + "="*80)
print("APPLYING CAPPING...")
print("="*80 + "\n")

for feature, rules in capping_rules.items():
    min_val = rules.get('min')
    max_val = rules.get('max')
    percentile = rules.get('percentile')
    
    # Cap training data
    below_train, above_train, total_train, pct_train, computed_max_train = cap_outliers(
        X_train, feature, min_val, max_val, percentile
    )
    
    # Use same max value for test data (if percentile was used)
    if computed_max_train is not None:
        max_val_test = computed_max_train
    else:
        max_val_test = max_val
    
    # Cap test data
    below_test, above_test, total_test, pct_test, _ = cap_outliers(
        X_test, feature, min_val, max_val_test, None
    )
    
    outlier_summary.append({
        'Feature': feature,
        'Train_Capped': total_train,
        'Train_Pct': pct_train,
        'Test_Capped': total_test,
        'Test_Pct': pct_test,
        'Min_Cap': min_val,
        'Max_Cap': computed_max_train if computed_max_train else max_val
    })
    
    print(f"✅ {feature}:")
    print(f"   Training - {total_train:,} values capped ({pct_train:.2f}%)")
    print(f"   Test - {total_test:,} values capped ({pct_test:.2f}%)")
    if min_val:
        print(f"   Min cap: {min_val}")
    if computed_max_train or max_val:
        print(f"   Max cap: {computed_max_train if computed_max_train else max_val:.2f}")
    print()

outlier_df = pd.DataFrame(outlier_summary)
print("\n" + "="*80)
print("OUTLIER CAPPING SUMMARY")
print("="*80)
display(outlier_df)


APPLYING CAPPING...

✅ age:
   Training - 14 values capped (0.01%)
   Test - 3 values capped (0.00%)
   Min cap: 18
   Max cap: 100.00

✅ DebtRatio:
   Training - 29,646 values capped (19.76%)
   Test - 19,942 values capped (19.65%)
   Max cap: 5.00

✅ RevolvingUtilizationOfUnsecuredLines:
   Training - 371 values capped (0.25%)
   Test - 243 values capped (0.24%)
   Max cap: 2.00

✅ NumberOfTime30-59DaysPastDueNotWorse:
   Training - 269 values capped (0.18%)
   Test - 215 values capped (0.21%)
   Max cap: 15.00

✅ NumberOfTimes90DaysLate:
   Training - 270 values capped (0.18%)
   Test - 217 values capped (0.21%)
   Max cap: 15.00

✅ NumberOfTime60-89DaysPastDueNotWorse:
   Training - 269 values capped (0.18%)
   Test - 214 values capped (0.21%)
   Max cap: 15.00

✅ NumberOfDependents:
   Training - 2 values capped (0.00%)
   Test - 2 values capped (0.00%)
   Max cap: 10.00


OUTLIER CAPPING SUMMARY


,Feature,Train_Capped,Train_Pct,Test_Capped,Test_Pct,Min_Cap,Max_Cap
0,age,14,0.0093,3,0.0030,18.0000,100.0000
1,DebtRatio,29646,19.7640,19942,19.6467,NaN,5.0000
2,RevolvingUtilizationOfUnsecuredLines,371,0.2473,243,0.2394,NaN,2.0000
3,NumberOfTime30-59DaysPastDueNotWorse,269,0.1793,215,0.2118,NaN,15.0000
4,NumberOfTimes90DaysLate,270,0.1800,217,0.2138,NaN,15.0000
5,NumberOfTime60-89DaysPastDueNotWorse,269,0.1793,214,0.2108,NaN,15.0000
6,NumberOfDependents,2,0.0013,2,0.0020,NaN,10.0000


## 3️⃣ Feature Engineering

### Strategy:
- Create **interaction features** from domain knowledge (credit scoring)
- Create **binary flags** to capture important thresholds
- Create **aggregation features** to simplify late payment signals
- Apply **before scaling** so new features are also scaled later

### New Features:
| Feature | Formula | Rationale |
|---------|---------|-----------|
| TotalLatePayments | Sum of 3 late payment columns | Aggregate delinquency signal |
| HasLatePayment | Binary: any late payment | Simplifies delinquency |
| IncomePerDependent | Income / (Dependents + 1) | Financial capacity per person |
| EstMonthlyDebt | DebtRatio × MonthlyIncome | Absolute debt amount |
| HighDebt | Flag: DebtRatio > 1 | Debt exceeds income |
| AgeBin | Age groups (6 bins) | Non-linear age effects |
| HighCreditUtil | Flag: Utilization > 100% | Maxed out credit |

In [10]:
print("="*80)
print("FEATURE ENGINEERING")
print("="*80)
print("\nCreating new features from domain knowledge...\n")

# --- 1. Total Late Payments (aggregate delinquency) ---
X_train['TotalLatePayments'] = (
    X_train['NumberOfTime30-59DaysPastDueNotWorse'] + 
    X_train['NumberOfTime60-89DaysPastDueNotWorse'] + 
    X_train['NumberOfTimes90DaysLate']
)
X_test['TotalLatePayments'] = (
    X_test['NumberOfTime30-59DaysPastDueNotWorse'] + 
    X_test['NumberOfTime60-89DaysPastDueNotWorse'] + 
    X_test['NumberOfTimes90DaysLate']
)
print("✅ TotalLatePayments = Sum of all 3 late payment types")

# --- 2. Has Any Late Payment (binary flag) ---
X_train['HasLatePayment'] = (X_train['TotalLatePayments'] > 0).astype(int)
X_test['HasLatePayment'] = (X_test['TotalLatePayments'] > 0).astype(int)
print("✅ HasLatePayment = Binary flag (any late payment)")

# --- 3. Income Per Dependent ---
X_train['IncomePerDependent'] = X_train['MonthlyIncome'] / (X_train['NumberOfDependents'] + 1)
X_test['IncomePerDependent'] = X_test['MonthlyIncome'] / (X_test['NumberOfDependents'] + 1)

# Cap at 99th percentile (fit on train only)
cap_income_dep = X_train['IncomePerDependent'].quantile(0.99)
X_train['IncomePerDependent'] = X_train['IncomePerDependent'].clip(upper=cap_income_dep)
X_test['IncomePerDependent'] = X_test['IncomePerDependent'].clip(upper=cap_income_dep)
print(f"✅ IncomePerDependent = MonthlyIncome / (Dependents + 1), capped at {cap_income_dep:,.0f}")

# --- 4. Estimated Monthly Debt ---
X_train['EstMonthlyDebt'] = X_train['DebtRatio'] * X_train['MonthlyIncome']
X_test['EstMonthlyDebt'] = X_test['DebtRatio'] * X_test['MonthlyIncome']

# Cap at 99th percentile (fit on train only)
cap_est_debt = X_train['EstMonthlyDebt'].quantile(0.99)
X_train['EstMonthlyDebt'] = X_train['EstMonthlyDebt'].clip(upper=cap_est_debt)
X_test['EstMonthlyDebt'] = X_test['EstMonthlyDebt'].clip(upper=cap_est_debt)
print(f"✅ EstMonthlyDebt = DebtRatio × MonthlyIncome, capped at {cap_est_debt:,.0f}")

# --- 5. High Debt Flag ---
X_train['HighDebt'] = (X_train['DebtRatio'] > 1).astype(int)
X_test['HighDebt'] = (X_test['DebtRatio'] > 1).astype(int)
print("✅ HighDebt = Flag for DebtRatio > 1 (debt exceeds income)")

# --- 6. Age Bins ---
age_bins = [0, 30, 40, 50, 60, 70, 120]
age_labels = [0, 1, 2, 3, 4, 5]
X_train['AgeBin'] = pd.cut(X_train['age'], bins=age_bins, labels=age_labels).astype(int)
X_test['AgeBin'] = pd.cut(X_test['age'], bins=age_bins, labels=age_labels).astype(int)
print("✅ AgeBin = Age groups (0:<30, 1:30-40, 2:40-50, 3:50-60, 4:60-70, 5:70+)")

# --- 7. High Credit Utilization Flag ---
X_train['HighCreditUtil'] = (X_train['RevolvingUtilizationOfUnsecuredLines'] > 1).astype(int)
X_test['HighCreditUtil'] = (X_test['RevolvingUtilizationOfUnsecuredLines'] > 1).astype(int)
print("✅ HighCreditUtil = Flag for credit utilization > 100%")

# --- Save FE capping values for reproducibility ---
fe_capping = {
    'IncomePerDependent_cap': cap_income_dep,
    'EstMonthlyDebt_cap': cap_est_debt
}

# --- Summary ---
new_features = ['TotalLatePayments', 'HasLatePayment', 'IncomePerDependent', 
                'EstMonthlyDebt', 'HighDebt', 'AgeBin', 'HighCreditUtil']

print(f"\n{'='*80}")
print(f"FEATURE ENGINEERING SUMMARY")
print(f"{'='*80}")
print(f"Original features: 10")
print(f"New features: {len(new_features)}")
print(f"Total features: {X_train.shape[1]}")
print(f"\nTraining shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

print(f"\n{'='*80}")
print("NEW FEATURES STATISTICS (Training Set)")
print(f"{'='*80}")
display(X_train[new_features].describe().round(2))

FEATURE ENGINEERING

Creating new features from domain knowledge...

✅ TotalLatePayments = Sum of all 3 late payment types
✅ HasLatePayment = Binary flag (any late payment)
✅ IncomePerDependent = MonthlyIncome / (Dependents + 1), capped at 17,280
✅ EstMonthlyDebt = DebtRatio × MonthlyIncome, capped at 27,000
✅ HighDebt = Flag for DebtRatio > 1 (debt exceeds income)
✅ AgeBin = Age groups (0:<30, 1:30-40, 2:40-50, 3:50-60, 4:60-70, 5:70+)
✅ HighCreditUtil = Flag for credit utilization > 100%

FEATURE ENGINEERING SUMMARY
Original features: 10
New features: 7
Total features: 17

Training shape: (150000, 17)
Test shape: (101503, 17)

NEW FEATURES STATISTICS (Training Set)


,TotalLatePayments,HasLatePayment,IncomePerDependent,EstMonthlyDebt,HighDebt,AgeBin,HighCreditUtil
count,150000.0000,150000.0000,150000.0000,150000.0000,150000.0000,150000.0000,150000.0000
mean,0.4800,0.2000,4411.8700,6642.1500,0.2300,2.6400,0.0200
std,2.1800,0.4000,3069.6000,9856.2100,0.4200,1.4300,0.1500
min,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
25%,0.0000,0.0000,2151.0000,766.3000,0.0000,2.0000,0.0000
50%,0.0000,0.0000,4000.0000,2102.7900,0.0000,3.0000,0.0000
75%,0.0000,0.0000,5400.0000,4797.4200,0.0000,4.0000,0.0000
max,45.0000,1.0000,17280.2000,27000.0000,1.0000,5.0000,1.0000


## 4️⃣ Train-Validation Split

### Strategy:
- **Split ratio**: 80-20 (training-validation)
- **Stratified**: Maintain class distribution
- **Random state**: 42 (reproducibility)

In [11]:
print("="*80)
print("TRAIN-VALIDATION SPLIT")
print("="*80)

# Split data
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

print(f"\nSplit results:")
print(f"  Training set:   {X_tr.shape}")
print(f"  Validation set: {X_val.shape}")

# Verify stratification
print("\n" + "="*80)
print("CLASS DISTRIBUTION VERIFICATION")
print("="*80)

def print_class_distribution(y, dataset_name):
    counts = y.value_counts().sort_index()
    pcts = y.value_counts(normalize=True).sort_index() * 100
    print(f"\n{dataset_name}:")
    print(f"  Class 0: {counts.iloc[0]:,} ({pcts.iloc[0]:.2f}%)")
    print(f"  Class 1: {counts.iloc[1]:,} ({pcts.iloc[1]:.2f}%)")
    print(f"  Ratio: {counts.iloc[0] / counts.iloc[1]:.2f}:1")

print_class_distribution(y_train, "Original Training Set")
print_class_distribution(y_tr, "After Split - Training")
print_class_distribution(y_val, "After Split - Validation")

print("\n✅ Stratification successful: Class distributions maintained!")
print("\nNote: Imbalance (13.96:1) will be handled in Phase 3 via class_weight='balanced'")

TRAIN-VALIDATION SPLIT

Split results:
  Training set:   (120000, 17)
  Validation set: (30000, 17)

CLASS DISTRIBUTION VERIFICATION

Original Training Set:
  Class 0: 139,974 (93.32%)
  Class 1: 10,026 (6.68%)
  Ratio: 13.96:1

After Split - Training:
  Class 0: 111,979 (93.32%)
  Class 1: 8,021 (6.68%)
  Ratio: 13.96:1

After Split - Validation:
  Class 0: 27,995 (93.32%)
  Class 1: 2,005 (6.68%)
  Ratio: 13.96:1

✅ Stratification successful: Class distributions maintained!

Note: Imbalance (13.96:1) will be handled in Phase 3 via class_weight='balanced'


## 5️⃣ Feature Scaling

### Strategy:
- **StandardScaler**: Fit on training split → transform all sets
- **Both versions saved**: scaled (for LR, SVM) + unscaled (for tree-based models)

In [12]:
print("="*80)
print("FEATURE SCALING")
print("="*80)

# Create StandardScaler - fit on TRAINING split
scaler = StandardScaler()

# Fit on X_tr (training split)
X_tr_scaled = pd.DataFrame(
    scaler.fit_transform(X_tr),
    columns=X_tr.columns,
    index=X_tr.index
)

# Transform validation set
X_val_scaled = pd.DataFrame(
    scaler.transform(X_val),
    columns=X_val.columns,
    index=X_val.index
)

# Transform test set
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print("\n✅ Scaling completed using StandardScaler")
print("   Scaler fit on: X_tr (training split)")
print("   Transformed: X_tr, X_val, X_test")

print(f"\n   X_tr_scaled:   {X_tr_scaled.shape}")
print(f"   X_val_scaled:  {X_val_scaled.shape}")
print(f"   X_test_scaled: {X_test_scaled.shape}")

# Verify scaling
print("\n" + "="*80)
print("SCALING VERIFICATION (Training Set)")
print("="*80)
scaling_check = pd.DataFrame({
    'Feature': X_tr_scaled.columns,
    'Mean': X_tr_scaled.mean().round(6).values,
    'Std': X_tr_scaled.std().round(6).values
})
print("\nScaled features should have mean≈0 and std≈1:")
display(scaling_check)

print("\n✅ Pipeline: Missing Values → Outliers → FE → Split → Scale")

FEATURE SCALING

✅ Scaling completed using StandardScaler
   Scaler fit on: X_tr (training split)
   Transformed: X_tr, X_val, X_test

   X_tr_scaled:   (120000, 17)
   X_val_scaled:  (30000, 17)
   X_test_scaled: (101503, 17)

SCALING VERIFICATION (Training Set)

Scaled features should have mean≈0 and std≈1:


,Feature,Mean,Std
0,RevolvingUtilizationOfUnsecuredLines,-0.0000,1.0000
1,age,0.0000,1.0000
2,NumberOfTime30-59DaysPastDueNotWorse,-0.0000,1.0000
3,DebtRatio,-0.0000,1.0000
4,MonthlyIncome,0.0000,1.0000
5,NumberOfOpenCreditLinesAndLoans,0.0000,1.0000
6,NumberOfTimes90DaysLate,-0.0000,1.0000
7,NumberRealEstateLoansOrLines,-0.0000,1.0000
8,NumberOfTime60-89DaysPastDueNotWorse,-0.0000,1.0000
9,NumberOfDependents,-0.0000,1.0000



✅ Pipeline: Missing Values → Outliers → FE → Split → Scale


## 6️⃣ Summary & Save Results

In [13]:
print("="*80)
print("PREPROCESSING SUMMARY")
print("="*80)

print("\n✅ COMPLETED STEPS:")
print("  1. Missing Values:")
print(f"     - MonthlyIncome: Imputed {missing_before_train['MonthlyIncome']:,} values (median=${imputed_value_income:,.2f})")
print(f"     - NumberOfDependents: Imputed {missing_before_train['NumberOfDependents']:,} values (median={imputed_value_dependents:.0f})")

print("\n  2. Outliers:")
total_capped = outlier_df['Train_Capped'].sum()
print(f"     - Total values capped: {total_capped:,}")
print(f"     - Features affected: {len(outlier_df)}")
print(f"     - DebtRatio capped at max=5 (was percentile 99=4979)")

print("\n  3. Feature Engineering:")
print(f"     - New features created: {len(new_features)}")
print(f"     - {', '.join(new_features)}")
print(f"     - Total features: {X_tr.shape[1]} (was 10)")

print("\n  4. Train-Validation Split:")
print(f"     - Training: {len(X_tr):,} samples")
print(f"     - Validation: {len(X_val):,} samples")
print(f"     - Split ratio: 80-20, Stratified: Yes")

print("\n  5. Feature Scaling:")
print(f"     - StandardScaler fit on X_tr")
print(f"     - Transformed: X_tr, X_val, X_test")

print("\n  6. Imbalanced Data:")
print(f"     - Ratio: 13.96:1 (93.32% Good vs 6.68% Bad)")
print(f"     - Strategy: class_weight='balanced' / scale_pos_weight (in Phase 3)")
print(f"     - Reason: Avoid synthetic samples from SMOTE that may introduce noise")

print("\n" + "="*80)
print("AVAILABLE DATASETS")
print("="*80)
print("\nFor tree-based models (no scaling needed):")
print(f"  - X_tr:       {X_tr.shape}")
print(f"  - X_val:      {X_val.shape}")
print(f"  - X_test:     {X_test.shape}")

print("\nFor linear models (scaled):")
print(f"  - X_tr_scaled:   {X_tr_scaled.shape}")
print(f"  - X_val_scaled:  {X_val_scaled.shape}")
print(f"  - X_test_scaled: {X_test_scaled.shape}")

print("\nTarget variables:")
print(f"  - y_tr:  {y_tr.shape}")
print(f"  - y_val: {y_val.shape}")

PREPROCESSING SUMMARY

✅ COMPLETED STEPS:
  1. Missing Values:
     - MonthlyIncome: Imputed 29,731 values (median=$5,400.00)
     - NumberOfDependents: Imputed 3,924 values (median=0)

  2. Outliers:
     - Total values capped: 30,841
     - Features affected: 7
     - DebtRatio capped at max=5 (was percentile 99=4979)

  3. Feature Engineering:
     - New features created: 7
     - TotalLatePayments, HasLatePayment, IncomePerDependent, EstMonthlyDebt, HighDebt, AgeBin, HighCreditUtil
     - Total features: 17 (was 10)

  4. Train-Validation Split:
     - Training: 120,000 samples
     - Validation: 30,000 samples
     - Split ratio: 80-20, Stratified: Yes

  5. Feature Scaling:
     - StandardScaler fit on X_tr
     - Transformed: X_tr, X_val, X_test

  6. Imbalanced Data:
     - Ratio: 13.96:1 (93.32% Good vs 6.68% Bad)
     - Strategy: class_weight='balanced' / scale_pos_weight (in Phase 3)
     - Reason: Avoid synthetic samples from SMOTE that may introduce noise

AVAILABLE DATASE

In [14]:
print("="*80)
print("SAVING PROCESSED DATA & OBJECTS")
print("="*80)

import os
output_dir = '../data/processed/'
os.makedirs(output_dir, exist_ok=True)

# === Save CSV files ===
print("\nSaving CSV files...")

# Unscaled data (for tree-based models)
X_tr.to_csv(output_dir + 'X_train.csv', index=False)
X_val.to_csv(output_dir + 'X_val.csv', index=False)
X_test.to_csv(output_dir + 'X_test.csv', index=False)

# Scaled data (for linear models)
X_tr_scaled.to_csv(output_dir + 'X_train_scaled.csv', index=False)
X_val_scaled.to_csv(output_dir + 'X_val_scaled.csv', index=False)
X_test_scaled.to_csv(output_dir + 'X_test_scaled.csv', index=False)

# Target variables
y_tr.to_csv(output_dir + 'y_train.csv', index=False, header=True)
y_val.to_csv(output_dir + 'y_val.csv', index=False, header=True)

print("✅ CSV files saved (8 files)")

# === Save preprocessing objects ===
print("\nSaving preprocessing objects...")

joblib.dump(income_imputer, output_dir + 'income_imputer.pkl')
joblib.dump(dependents_imputer, output_dir + 'dependents_imputer.pkl')
joblib.dump(scaler, output_dir + 'scaler.pkl')
joblib.dump(capping_rules, output_dir + 'capping_rules.pkl')
joblib.dump(fe_capping, output_dir + 'fe_capping.pkl')

print("✅ Preprocessing objects saved:")
print("   - income_imputer.pkl")
print("   - dependents_imputer.pkl")
print("   - scaler.pkl")
print("   - capping_rules.pkl")
print("   - fe_capping.pkl (Feature Engineering capping values)")

# === Save summary reports ===
print("\nSaving summary reports...")

report_dir = '../reports/[2] Ket qua Preprocessing/'
os.makedirs(report_dir, exist_ok=True)

missing_summary_before.to_csv(report_dir + 'missing_values_summary.csv', index=False)
outlier_df.to_csv(report_dir + 'outlier_capping_summary.csv', index=False)
scaling_check.to_csv(report_dir + 'scaling_verification.csv', index=False)

print("✅ Summary reports saved")

print("\n" + "="*80)
print("✅ ALL DATA AND OBJECTS SAVED SUCCESSFULLY!")
print("="*80)

SAVING PROCESSED DATA & OBJECTS

Saving CSV files...
✅ CSV files saved (8 files)

Saving preprocessing objects...
✅ Preprocessing objects saved:
   - income_imputer.pkl
   - dependents_imputer.pkl
   - scaler.pkl
   - capping_rules.pkl
   - fe_capping.pkl (Feature Engineering capping values)

Saving summary reports...
✅ Summary reports saved

✅ ALL DATA AND OBJECTS SAVED SUCCESSFULLY!


In [15]:
# Create data quality report
quality_report = pd.DataFrame({
    'Metric': [
        'Original Features',
        'New Features (FE)',
        'Total Features',
        'Total Samples (Training)',
        'Total Samples (Validation)',
        'Total Samples (Test)',
        'Missing Values (Before)',
        'Missing Values (After)',
        'Outliers Capped',
        'DebtRatio Max Cap',
        'Features Scaled',
        'Class Imbalance',
        'Imbalance Strategy'
    ],
    'Value': [
        "10",
        f"{len(new_features)}",
        f"{X_tr.shape[1]}",
        f"{len(X_tr):,}",
        f"{len(X_val):,}",
        f"{len(X_test):,}",
        f"{missing_before_train.sum():,}",
        "0",
        f"{total_capped:,}",
        "5 (was 4,979)",
        "All (StandardScaler)",
        "13.96:1",
        "class_weight='balanced' (in Phase 3)"
    ]
})

print("="*80)
print("DATA QUALITY REPORT")
print("="*80)
display(quality_report)

quality_report.to_csv(report_dir + 'data_quality_report.csv', index=False)
print("\n✅ Data quality report saved!")

DATA QUALITY REPORT


,Metric,Value
0,Original Features,10
1,New Features (FE),7
2,Total Features,17
3,Total Samples (Training),"120,000"
4,Total Samples (Validation),"30,000"
5,Total Samples (Test),"101,503"
6,Missing Values (Before),"33,655"
7,Missing Values (After),0
8,Outliers Capped,"30,841"
9,DebtRatio Max Cap,"5 (was 4,979)"



✅ Data quality report saved!


## ✅ Preprocessing Complete!

### Pipeline:
```
Raw Data → Missing Values → Outliers (Capping) → Feature Engineering (+7 features)
         → Train-Val Split (80/20) → Scaling (StandardScaler)
```

### Imbalanced Data:
- **Khong dung SMOTE** — thay vao do, xu ly trong Phase 3 bang `class_weight='balanced'` / `scale_pos_weight`
- **Ly do**: class_weight dieu chinh loss function truc tiep, khong tao mau tong hop nen tranh duoc nhieu tu synthetic samples va giu nguyen phan phoi goc cua data

### Files Generated:
- **CSV files**: 8 files in `data/processed/` (6 feature sets + 2 targets)
- **Preprocessing objects**: 5 pkl files
- **Reports**: 4 files in `reports/[2] Ket qua Preprocessing/`

### Next Steps:
1. **Phase 3**: Model Building (train 5 ML models voi class_weight='balanced')
2. **Phase 4**: Model Evaluation & Comparison

In [16]:
print(f"Preprocessing completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\n🎉 Ready for Phase 3: Model Building!")

Preprocessing completed at: 2026-03-14 22:11:05

🎉 Ready for Phase 3: Model Building!
